# **Loan Prediction**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# **1.Preprocessing Steps**

Read the csv data using pandas pandas.read_csv(path) function

In [ ]:
df = pd.read_csv("/kaggle/input/mydemodata/train_loan.csv")

Visualize the first 5 observations using dataframe.head() function. You can also pass the number of rows/observations inside head function as required as suppose you want to visualize first 10 observations -10 - dataframe_name.head(10)

In [ ]:
df.head()

In [ ]:
df['Education'].unique()

Check the shape of the data using .shape function

In [ ]:
df.shape

We have 614 observations/rows and 13 attributes.

# 1.1 Filling all missing values.

Check total missing values for each attribute.

In [ ]:
df.isnull().sum()

Find the data types of the variables.

In [ ]:
df.dtypes

int and float denote numerical data and object denotes categorical data.

# Some Analytical visualizations 

In [ ]:
import cv2 as cv

In [ ]:
from IPython.display import Image
import os
!ls ../input/sheet1

In [ ]:
Image("/kaggle/input/sheet1/Sheet 1 (1).png")

The above figure shows the Average Applicant income and corresponding Loan status for each Gender.

In [ ]:
Image("/kaggle/input/sheet2/Sheet 2.png")

Above Figure shows what is the Area wise Average Loan Amount

In [ ]:
Image("/kaggle/input/sheet3/Sheet 3.png")

Above figure shows At What Average Loan Amount , Loan is passed( indicated with Y) or not (N) for each Area.

# Categorical Data: Mode imputation

In [ ]:
df['Gender'].fillna(df['Gender'].mode()[0],inplace = True)
df['Married'].fillna(df['Married'].mode()[0],inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0],inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0],inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0],inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0],inplace=True)

# Numerical Data: Mean imputation

In [ ]:
df['LoanAmount'].fillna(df['LoanAmount'].mean(),inplace=True)

In [ ]:
df.isnull().sum()

# 1.2 Converting categories to numbers 

1.2.1Convert object variables into numerical using map function.

In [ ]:
df['Gender'] = df['Gender'].map({"Male":0,'Female':1})

1.2.2Categorical to numerical using LabelEncoder 

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Dependents'] = le.fit_transform(df['Dependents'])
df['Property_Area'] = le.fit_transform(df['Property_Area'])
df['Married'] = le.fit_transform(df['Married'])
df['Self_Employed'] = le.fit_transform(df['Self_Employed'])
df['Loan_Status'] = le.fit_transform(df['Loan_Status'])
df['Education'] = le.fit_transform(df['Education'])

In [ ]:
df.head()

In [ ]:
df.dtypes

# 1.3 Scale down all variables to same range 

1.3.1 Min Max Normalization: Scales down all vaues between 0 and 1 

In [ ]:
df['ApplicantIncome'] = (df['ApplicantIncome'] - df['ApplicantIncome'].min())/(df['ApplicantIncome'].max()-df['ApplicantIncome'].min())

In [ ]:
for i in df.columns[1:]:
    df[i] = (df[i] - df[i].min())/ (df[i].max() - df[i].min())

In [ ]:
df.head()

**saving the Preprocessed data**

In [ ]:
df.to_csv('Loan_data_preprocessed',index=False)

# **2.Load the Prprocessed dataset**

# Steps to build a Neural Network:
1. Load Preprocessed data
2. Create Train and Test set
3. Define model architecture
4. define the Loss function and optimizers
5. Compile the model
6. Train the model
7. Evaluate model performance

In [ ]:
df = pd.read_csv("/kaggle/working/Loan_data_preprocessed")

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop('Loan_ID',axis=1)

In [ ]:
df.shape

**Seperating dependent and independent variables.**

In [ ]:
x = df.drop('Loan_Status',axis = 1)
y = df['Loan_Status']

In [ ]:
x.shape,y.shape

# **3.Creating Training and Validation set**

In [ ]:
# stratify so that, the distribution of class will be same in train and test data
# test size = 0.2 sets 20% data as test data and 80% data as train data
# random state = 42 ensures same plit in each run of the code, i.e. we can reproduce the results
x_train,x_test,y_train,y_test = train_test_split(x,y,stratify=df['Loan_Status'],test_size=0.2,random_state=42)

In [ ]:
(x_train.shape, x_test.shape), (y_train.shape,y_test.shape)

# **4.Defining Neural Network Model Architecture**

In [ ]:
from keras.models import Sequential

**Define the Model layers**

In [ ]:
from keras.layers import InputLayer, Dense

In [ ]:
# defining the input neurons 
input_neurons = x_train.shape[1]

In [ ]:
# Since , it is binary classification problem , we will use only 1 output neuron
output_neurons = 1

In [ ]:
# Defining hidden layers and hidden neurons
# these are hyperparameters and we can choose any number of neurons and hidden layers
number_of_hidden_layers = 2
neurons_hidden_layer1 = 10
neurons_hidden_layer2 = 5

I am using Relu as Activation function for hidden layers and , since it is a Binary Class classification problem, Sigmoid as final Activation function.

In [ ]:
model = Sequential()
model.add(InputLayer(input_shape=input_neurons,))
model.add(Dense(units=neurons_hidden_layer1,activation='relu'))
model.add(Dense(units=neurons_hidden_layer2,activation='relu'))
model.add(Dense(units=output_neurons,activation='sigmoid'))

In [ ]:
model.summary()

1. Number of parameters in first Hidden layer = No. of input neurons * No. of neurons in 1st Hidden layer + No.of neurons in 1st hidden layer (11*10+10 = 120)
2. Number of parameters in 2nd Hidden layer = No. of neurons in 1st Hidden Layer * No. of neurons in 2nd hidden layer + No.of neurons in 2nd hidden layer (10*5+5 = 55)
3. Noumber of parametsrs in Output Layer = No. of neurons in 2nd Hidden layer * No. of output neurons + No. of output neurons (5*1 + 1 = 6)

**Compile the model (define Loss function and optimizer)**

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

**Training the model.**

In [ ]:
model_history = model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=50)

# **5.Evaluate model performance**

In [ ]:
predicted_probabilities = model.predict(x_test)
# Convert probabilities to predicted classes (0 or 1)
predicted_classes = np.round(predicted_probabilities)
accuracy_score = accuracy_score(y_test,predicted_classes)
print("Accuracy: ",accuracy_score)

# **6.Visualize model performance**

In [ ]:
plt.plot(model_history.history['loss'])
plt.plot(model_history.history['val_loss'])
plt.title("Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("loss")
plt.legend(['train','validation'],loc='lower right')
plt.show()

In [ ]:
plt.plot(model_history.history['accuracy'])
plt.plot(model_history.history['val_accuracy'])
plt.title("accuracy Curve")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(['train','validation'],loc='lower right')
plt.show()